# Stroke Risk — Hyperparameter Tuning

We tune both models with **randomized search**, optimising **PR-AUC** (our lead metric for this rare event).

**Leak-free discipline:**
- the search runs with stratified cross-validation on the **training set only**;
- we compare **default vs tuned** on the **validation** set;
- the **test set is scored once**, at the end.

## Setup

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stroke_risk import data, evaluate, model, tune
from stroke_risk import plotting as pl
from stroke_risk.features import split_features_target

pl.set_theme()
warnings.filterwarnings('ignore')

splits = data.split_data(data.load_raw())
X_train, y_train = split_features_target(splits.train)
X_val, y_val = split_features_target(splits.val)
X_test, y_test = split_features_target(splits.test)

## 1. Randomized search (training set only)

Each model is searched over its own space; the score is cross-validated PR-AUC.

In [ ]:
specs = tune.build_search_specs(y_train)
searches = {
    name: tune.tune(pipe, space, X_train, y_train)
    for name, (pipe, space) in specs.items()
}
pd.DataFrame({n: {'best CV PR-AUC': s.best_score_} for n, s in searches.items()}).T.round(3)

In [ ]:
# Best hyperparameters found for each model
for name, s in searches.items():
    print(name)
    for key, val in s.best_params_.items():
        pretty = round(val, 4) if isinstance(val, float) else val
        print(f'   {key.replace("clf__", "")}: {pretty}')

## 2. Tuned vs default on validation

Did tuning actually help? We compare on validation data the model has not been tuned against.

In [ ]:
default_models = model.build_models(y_train)
rows = {}
for name in specs:
    fitted = {
        'default': default_models[name].fit(X_train, y_train),
        'tuned': searches[name].best_estimator_,  # already refit on train
    }
    for label, mdl in fitted.items():
        proba = mdl.predict_proba(X_val)[:, 1]
        thr = evaluate.choose_threshold(y_val, proba)
        rows[(name, label)] = evaluate.summarize(y_val, proba, thr)

pd.DataFrame(rows).T[['PR-AUC', 'ROC-AUC', 'recall', 'precision', 'f1']].round(3)

In [ ]:
names = list(specs)
default_pr = [rows[(n, 'default')]['PR-AUC'] for n in names]
tuned_pr = [rows[(n, 'tuned')]['PR-AUC'] for n in names]
x = np.arange(len(names))
w = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
b1 = ax.bar(x - w / 2, default_pr, w, color=pl.MUTED, label='default')
b2 = ax.bar(x + w / 2, tuned_pr, w, color=pl.ACCENT, label='tuned')
ax.bar_label(b1, fmt='%.3f', padding=3, color=pl.SUBTLE, fontsize=9)
ax.bar_label(b2, fmt='%.3f', padding=3, color=pl.SUBTLE, fontsize=9)
ax.set_xticks(x, names)
ax.set_yticks([])
ax.grid(False)
ax.legend(loc='upper right')
pl.despine(ax, left=True)
pl.add_titles(ax, 'Does tuning help?', 'Validation PR-AUC: default vs tuned')
plt.show()

## 3. Final model on the held-out test set

We pick the best **tuned** model by validation PR-AUC, then score the test set once.

In [ ]:
winner = max(names, key=lambda n: rows[(n, 'tuned')]['PR-AUC'])
best_model = searches[winner].best_estimator_

proba_val = best_model.predict_proba(X_val)[:, 1]
best_thr = evaluate.choose_threshold(y_val, proba_val)
proba_test = best_model.predict_proba(X_test)[:, 1]

print(f'Winner: {winner} (tuned)   threshold = {best_thr:.3f}')
evaluate.metrics_table({winner: evaluate.summarize(y_test, proba_test, best_thr)})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
evaluate.plot_pr_curve(axes[0], y_test, proba_test, f'{winner} (tuned)')
axes[0].set_title('Precision–Recall', loc='left', fontsize=12, fontweight='bold', color=pl.INK)
evaluate.plot_roc_curve(axes[1], y_test, proba_test, f'{winner} (tuned)')
axes[1].set_title('ROC', loc='left', fontsize=12, fontweight='bold', color=pl.INK)
plt.tight_layout()
plt.show()

In [ ]:
y_pred_test = (proba_test >= best_thr).astype(int)
fig, ax = plt.subplots(figsize=(5, 4.5))
evaluate.plot_confusion_matrix(ax, y_test, y_pred_test)
pl.add_titles(ax, f'Tuned {winner} on the test set', 'Confusion matrix at the chosen threshold')
plt.show()

## Conclusions

- Tuning is done **correctly**: searched on train, compared on validation, tested once.
- The winning **tuned pipeline** is the artefact we will save and deploy next.
- On this dataset the ceiling is modest (PR-AUC stays low) — an honest reflection that stroke is hard to predict from these features alone.